# BEER thumbnails

This notebook generates the thumbnails used in the BEER user guide.

In [ ]:
import plopp as pp
import scipp as sc

from ess.beer import BeerModMcStasWorkflowKnownPeaks, BeerPowderMcStasWorkflow
from ess.beer import data as beer_data
from ess.beer.types import DetectorBank, DHKLList
from ess.powder import RunNormalization
from ess.powder.types import (
    DspacingBins,
    ElasticCoordTransformGraph,
    EmptyCanRun,
    EmptyCanSubtractedIofDspacing,
    Filename,
    KeepEvents,
    MaskedDetectorIDs,
    SampleRun,
    TofMask,
    TwoThetaMask,
    UncertaintyBroadcastMode,
    VanadiumRun,
    WavelengthDetector,
    WavelengthMask,
)


In [ ]:
modulation_workflow = BeerModMcStasWorkflowKnownPeaks()
modulation_workflow[Filename[SampleRun]] = beer_data.mcstas_silicon_new_model(9)
modulation_workflow[DetectorBank] = DetectorBank.south
modulation_workflow[DHKLList] = beer_data.silicon_peaks_array()
modulation_result = modulation_workflow.compute(
    (WavelengthDetector[SampleRun], ElasticCoordTransformGraph[SampleRun])
)
modulation_data = modulation_result[WavelengthDetector[SampleRun]].transform_coords(
    'two_theta', graph=modulation_result[ElasticCoordTransformGraph[SampleRun]]
)
modulation_histogram = modulation_data.hist(
    two_theta=400, event_time_offset=1000
)

In [ ]:
def set_foreground(fig, color: str) -> None:
    for ax in fig.fig.axes:
        ax.tick_params(colors=color)
        ax.xaxis.label.set_color(color)
        ax.yaxis.label.set_color(color)
        for spine in ax.spines.values():
            spine.set_color(color)


def modulation_plot():
    return pp.plot(
        modulation_histogram,
        figsize=(3, 2.5),
        norm='log',
        cmin=1.0e-3,
        xlabel=r'$t$ [s]',
        ylabel=r'$2\theta$ [rad]',
    )

In [ ]:
fig = modulation_plot()
fig.save(
    "../../docs/_static/thumbnails/beer_mcstas_light.svg",
    transparent=True,
)
fig

In [ ]:
fig = modulation_plot()
set_foreground(fig, 'white')
fig.save(
    "../../docs/_static/thumbnails/beer_mcstas_dark.svg",
    transparent=True,
)
fig

## Powder diffraction workflow

In [ ]:
powder_workflow = BeerPowderMcStasWorkflow(
    run_norm=RunNormalization.monitor_histogram
)
powder_workflow[Filename[SampleRun]] = (
    beer_data.mcstas_powder_silicon_in_vanadium_can()
)
powder_workflow[Filename[VanadiumRun]] = beer_data.mcstas_powder_vanadium()
powder_workflow[Filename[EmptyCanRun]] = beer_data.mcstas_powder_empty_can()
powder_workflow[DetectorBank] = DetectorBank.north
powder_workflow[DspacingBins] = sc.linspace(
    'dspacing', 0.6, 2.0, 701, unit='angstrom'
)

for run_type in (SampleRun, VanadiumRun, EmptyCanRun):
    powder_workflow[KeepEvents[run_type]] = KeepEvents(True)

powder_workflow[MaskedDetectorIDs] = MaskedDetectorIDs({})
powder_workflow[UncertaintyBroadcastMode] = UncertaintyBroadcastMode.drop
powder_workflow[TwoThetaMask] = None
powder_workflow[TofMask] = None
powder_workflow[WavelengthMask] = None

In [ ]:
powder_pattern = powder_workflow.compute(
    EmptyCanSubtractedIofDspacing
).hist()
powder_pattern = powder_pattern.assign(
    sc.where(
        powder_pattern.masks['zero_vanadium'],
        sc.scalar(float('nan'), unit=powder_pattern.unit),
        powder_pattern.data,
    )
)

In [ ]:
def powder_diffraction_plot():
    return pp.plot(
        powder_pattern,
        figsize=(3, 2.5),
        xmin=0.6,
        xmax=2.0,
        xlabel=r'$d$ [Å]',
        ylabel=r'$I(d)$',
        errorbars=False,
        linewidth=1.2,
    )

In [ ]:
fig = powder_diffraction_plot()
fig.save(
    "../../docs/_static/thumbnails/beer_powder_reduction_light.svg",
    transparent=True,
)
fig

In [ ]:
fig = powder_diffraction_plot()
set_foreground(fig, 'white')
fig.save(
    "../../docs/_static/thumbnails/beer_powder_reduction_dark.svg",
    transparent=True,
)
fig